# Semana 7 · Consultas SQL y su equivalente en pandas

**Caso:** Spotify — capacidad de streaming y curaduría regional · **Autor:** Julián Andrés Solano Ledesma

Este cuaderno:

1. carga las tablas normalizadas de la Semana 6 en una base **SQLite** en memoria, con el mismo esquema (PK, FK y `CHECK`);
2. ejecuta las 3 consultas de `consultas-spotify.sql`: **WHERE**, **JOIN** y **GROUP BY**;
3. reproduce la consulta **GROUP BY** en **pandas** (`merge` + `groupby`) y comprueba por código que ambos resultados son idénticos.

## 1. Carga de datos y revisión previa (tipos y nulos)

In [1]:
import re
import sqlite3
from pathlib import Path

import pandas as pd

DATOS = Path("datos")
ESQUEMA = Path("../../06-week/02-optional-activity/esquema-spotify.sql")
TABLAS = ["pais", "genero", "plan_suscripcion", "artista", "album", "metrica_streaming"]

df = {t: pd.read_csv(DATOS / f"{t}.csv") for t in TABLAS}

con = sqlite3.connect(":memory:")
con.executescript(ESQUEMA.read_text(encoding="utf-8"))
for t in TABLAS:                       # catálogos primero, para respetar las FK
    df[t].to_sql(t, con, if_exists="append", index=False)

pd.DataFrame({
    "filas": {t: len(df[t]) for t in TABLAS},
    "nulos": {t: int(df[t].drop(columns="genero_principal", errors="ignore").isna().sum().sum()) for t in TABLAS},
})

,filas,nulos
pais,20,0
genero,10,0
plan_suscripcion,2,0
artista,15,0
album,15,0
metrica_streaming,500,0


In [2]:
# Tipos de la tabla de hechos: las medidas deben ser numéricas antes de agregarlas
df["metrica_streaming"].dtypes.to_frame("tipo").T

,metric_id,country_id,album_id,genero_id,plan_id,oyentes_mensuales_m,streams_totales_m,horas_totales_m,duracion_prom_min,streams_30d_m,skip_rate_pct
tipo,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64


In [3]:
# Las consultas se leen directamente del archivo .sql entregado (una por bloque Q1, Q2, Q3)
sql = Path("consultas-spotify.sql").read_text(encoding="utf-8")
sin_comentarios = re.sub(r"--.*", "", sql)
Q1, Q2, Q3 = [q.strip() for q in sin_comentarios.split(";") if q.strip()]
print(Q3)

SELECT p.nombre                         AS pais,
       COUNT(*)                         AS n_registros,
       ROUND(SUM(m.streams_30d_m), 2)   AS streams_30d_total_m,
       ROUND(AVG(m.streams_30d_m), 2)   AS streams_30d_prom_m,
       ROUND(AVG(m.skip_rate_pct), 2)   AS skip_rate_prom_pct
FROM   metrica_streaming m
JOIN   pais p ON p.country_id = m.country_id
GROUP  BY p.nombre
ORDER  BY streams_30d_total_m DESC


## 2. Q1 · WHERE — alta demanda reciente y alto abandono

**Pregunta:** ¿qué registros país–álbum tienen a la vez más de 150 M de streams en 30 días y un *skip rate* mayor al 30 %?

In [4]:
q1 = pd.read_sql(Q1, con)
print(f"{len(q1)} de {len(df['metrica_streaming'])} registros ({len(q1) / len(df['metrica_streaming']):.1%})")
q1.head(5)

24 de 500 registros (4.8%)


,metric_id,country_id,album_id,streams_30d_m,skip_rate_pct
0,14,6,13,163.39,39.14
1,370,1,6,155.45,38.61
2,412,8,3,170.26,37.30
3,15,14,12,192.91,37.23
4,21,17,15,193.83,37.13


In [5]:
# Solo para la lectura: nombres del registro con mayor abandono
top = q1.iloc[0]
pais = df["pais"].set_index("country_id").loc[top.country_id, "nombre"]
alb = df["album"].set_index("album_id").loc[top.album_id]
art = df["artista"].set_index("artist_id").loc[alb.artist_id, "nombre"]
print(f"Mayor abandono: registro {int(top.metric_id)} · {pais} · {art} · {alb.titulo} · "
      f"{top.skip_rate_pct} % de skip con {top.streams_30d_m} M de streams")

Mayor abandono: registro 14 · Germany · SZA · SOS · 39.14 % de skip con 163.39 M de streams


## 3. Q2 · JOIN — las combinaciones con más demanda reciente

**Pregunta:** ¿qué combinaciones país–artista–álbum concentran más streams en los últimos 30 días?

In [6]:
q2 = pd.read_sql(Q2, con)
q2

,pais,artista,album,plan,streams_30d_m
0,Italy,Taylor Swift,1989 (Taylor's Version),Free,200.00
1,Brazil,Ed Sheeran,Autumn Variations,Free,199.91
2,Argentina,Ariana Grande,Eternal Sunshine,Free,198.06
3,India,BTS,Proof,Premium,197.84
4,South Korea,SZA,SOS,Premium,197.83
5,France,Bad Bunny,Nadie Sabe Lo Que Va a Pasar Mañana,Free,197.08
6,United States,BLACKPINK,BORN PINK,Free,196.16
7,France,Drake,For All The Dogs,Free,196.15
8,Brazil,Ariana Grande,Eternal Sunshine,Free,194.95
9,Argentina,The Weeknd,After Hours,Free,194.82


## 4. Q3 · GROUP BY — demanda y abandono por país

**Pregunta:** ¿cuánta demanda reciente acumula cada país y cuánto abandono promedio tiene?

In [7]:
q3_sql = pd.read_sql(Q3, con)
q3_sql

,pais,n_registros,streams_30d_total_m,streams_30d_prom_m,skip_rate_prom_pct
0,Italy,34,3227.25,94.92,22.37
1,South Korea,31,3171.34,102.30,20.97
2,Brazil,30,3063.11,102.10,20.59
3,South Africa,32,3024.99,94.53,22.07
4,Sweden,30,3023.20,100.77,21.79
5,Russia,31,2886.25,93.10,21.90
6,Argentina,27,2834.22,104.97,22.62
7,Canada,25,2589.97,103.60,20.52
8,Japan,23,2471.66,107.46,22.76
9,France,23,2462.71,107.07,19.08


## 5. Equivalente en pandas de Q3

Cada cláusula SQL tiene su equivalente en pandas:

| SQL | pandas |
|---|---|
| `JOIN pais ON ...` | `merge(..., on="country_id")` |
| `GROUP BY p.nombre` | `groupby("nombre")` |
| `COUNT`, `SUM`, `AVG` | `agg(count, sum, mean)` |
| `ROUND(..., 2)` | `round(2)` |
| `ORDER BY ... DESC` | `sort_values(..., ascending=False)` |

In [8]:
m, p = df["metrica_streaming"], df["pais"]

q3_pandas = (
    m.merge(p, on="country_id")                       # JOIN pais
     .groupby("nombre")                               # GROUP BY p.nombre
     .agg(n_registros=("metric_id", "count"),         # COUNT(*)
          streams_30d_total_m=("streams_30d_m", "sum"),
          streams_30d_prom_m=("streams_30d_m", "mean"),
          skip_rate_prom_pct=("skip_rate_pct", "mean"))
     .round(2)                                        # ROUND(..., 2)
     .sort_values("streams_30d_total_m", ascending=False)   # ORDER BY ... DESC
     .reset_index()
     .rename(columns={"nombre": "pais"})
)
q3_pandas

,pais,n_registros,streams_30d_total_m,streams_30d_prom_m,skip_rate_prom_pct
0,Italy,34,3227.25,94.92,22.37
1,South Korea,31,3171.34,102.30,20.97
2,Brazil,30,3063.11,102.10,20.59
3,South Africa,32,3024.99,94.53,22.07
4,Sweden,30,3023.20,100.77,21.79
5,Russia,31,2886.25,93.10,21.90
6,Argentina,27,2834.22,104.97,22.62
7,Canada,25,2589.97,103.60,20.52
8,Japan,23,2471.66,107.46,22.76
9,France,23,2462.71,107.07,19.08


In [9]:
pd.testing.assert_frame_equal(q3_sql, q3_pandas, check_dtype=False)
print(f"OK: SQL y pandas devuelven el mismo resultado en los {len(q3_sql)} países y las 5 columnas.")

OK: SQL y pandas devuelven el mismo resultado en los 20 países y las 5 columnas.


## 6. Lectura del resultado de Q3

In [10]:
lider = q3_sql.iloc[0]
mayor_prom = q3_sql.sort_values("streams_30d_prom_m", ascending=False).iloc[0]
mayor_skip = q3_sql.sort_values("skip_rate_prom_pct", ascending=False).iloc[0]
rango = q3_sql["skip_rate_prom_pct"].agg(["min", "max"])

print(f"Mayor demanda total:      {lider.pais} ({lider.streams_30d_total_m:,.2f} M en {lider.n_registros} registros)")
print(f"Mayor demanda por registro: {mayor_prom.pais} ({mayor_prom.streams_30d_prom_m} M)")
print(f"Mayor abandono promedio:  {mayor_skip.pais} ({mayor_skip.skip_rate_prom_pct} %)")
print(f"Rango del skip promedio:  {rango['min']} % a {rango['max']} %")
print(f"Correlación registros–total: {q3_sql['n_registros'].corr(q3_sql['streams_30d_total_m']):.2f}")
con.close()

Mayor demanda total:      Italy (3,227.25 M en 34 registros)
Mayor demanda por registro: United Kingdom (110.7 M)
Mayor abandono promedio:  Indonesia (23.03 %)
Rango del skip promedio:  16.06 % a 23.03 %
Correlación registros–total: 0.94
